In [ ]:
import scanpy as sc
import spatialdata_plot
import spatialdata_io as io
import spatialdata as sd
import numpy as np
import matplotlib.pyplot as plt
import scprep
from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# Figure 6

In [ ]:
data_dict={}
for sample in ["B59460"  ,"P38685","P52407","P6055125","P62560"]:
    data_dict[sample] = sc.read_h5ad(f"/QRISdata/Q1851/Xiao/Working_project/MB/PROCESSED/{sample}_processed_v4.h5ad")
 

In [ ]:
import pandas as pd
from tqdm import tqdm

# Initialize list to store all results
all_results = []

for sample in ["B59460", "P38685", "P52407", "P6055125", "P62560"]:
    print(f"\nProcessing {sample}...")
    adata = data_dict[sample]
    
    # preprocess
    expr_data = adata.X
    expr_data = scprep.filter.filter_empty_cells(expr_data)
    expr_data = scprep.normalize.library_size_normalize(expr_data, rescale=100)
    expr_data = scprep.transform.sqrt(expr_data)
    
    # Get pool of genes
    pool = [gene for gene in adata.var_names if gene not in ("TP53", "MKI67")]
    
    # Pre-extract TP53 and MKI67 data
    tp53_idx = adata.var_names.get_loc("TP53")
    mki67_idx = adata.var_names.get_loc("MKI67")
    y_tp53 = expr_data[:, tp53_idx].A.flatten() if hasattr(expr_data, 'A') else expr_data[:, tp53_idx]
    y_mki67 = expr_data[:, mki67_idx].A.flatten() if hasattr(expr_data, 'A') else expr_data[:, mki67_idx]
    
    # Calculate DREMI scores for this sample
    for gene in tqdm(pool, desc=f"Calculating DREMI for {sample}"):
        gene_idx = adata.var_names.get_loc(gene)
        x = expr_data[:, gene_idx].A.flatten() if hasattr(expr_data, 'A') else expr_data[:, gene_idx]
        
        score_tp53 = scprep.stats.knnDREMI(x, y_tp53, k=10, n_bins=40, n_jobs=11, plot=False)
        score_mki67 = scprep.stats.knnDREMI(x, y_mki67, k=10, n_bins=40, n_jobs=11, plot=False)
        
        all_results.append({
            'Sample': sample,
            'Gene': gene,
            'DREMI to TP53': score_tp53,
            'DREMI to MKI67': score_mki67
        })

# Create DataFrame from all results
dremi_df = pd.DataFrame(all_results)

# Optional: Sort by sample and DREMI score
dremi_df = dremi_df.sort_values(['Sample', 'DREMI to TP53'], ascending=[True, False])

# Display summary
print("\nFinal table shape:", dremi_df.shape)
print("\nFirst few rows:")
print(dremi_df.head(10))

# Save to CSV
dremi_df.to_csv('all_samples_dremi_scores.csv', index=False)
print("\nSaved to all_samples_dremi_scores.csv")

In [ ]:
import csv
with open('all_samples_dremi_scores.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    # Write header
    writer.writerow(dremi_df.columns.tolist())
    # Write data
    for index, row in dremi_df.iterrows():
        writer.writerow(row.values.tolist())

# data used to plot with excel 